In [6]:
from pathlib import Path
import pandas as pd

def find_project_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/raw from " + str(path))

RAW_DIR = find_project_root() / "data" / "raw"

def read_xpt(period, name):
    return pd.read_sas(RAW_DIR / period / f"{name}.xpt", format="xport").round(6)

# Expected variables per component (D-006)
EXPECTED = {
    "DEMO":   ["RIAGENDR", "RIDAGEYR"],
    "BMX":    ["BMXBMI", "BMXWAIST"],
    "SMQ":    ["SMQ020", "SMQ040"],
    "PAQ":    ["PAQ605", "PAQ620", "PAQ635", "PAQ650", "PAQ665"],
    "DIQ":    ["DIQ010"],
    "MCQ":    ["MCQ300A", "MCQ300C"],
    "GHB":    ["LBXGH"],
    "TCHOL":  ["LBXTC"],
    "HDL":    ["LBDHDD"],
    "BIOPRO": ["LBXSCR"],
}
FILE_NAME = {
    "2017-2020": lambda c: f"P_{c}",
    "2021-2023": lambda c: f"{c}_L",
}

rows = []
for comp, cols in EXPECTED.items():
    for period, fname in FILE_NAME.items():
        df = read_xpt(period, fname(comp))
        upper = {c.upper(): c for c in df.columns}
        for col in cols:
            rows.append({"component": comp, "variable": col, "period": period,
                         "found": col in upper})

check = pd.DataFrame(rows).pivot_table(index=["component", "variable"],
                                       columns="period", values="found", aggfunc="first")
print(check)

# For any component with a missing variable, list what the file actually contains
for comp, cols in EXPECTED.items():
    for period, fname in FILE_NAME.items():
        df = read_xpt(period, fname(comp))
        upper = {c.upper() for c in df.columns}
        if not all(c in upper for c in cols):
            print(f"\n{fname(comp)} columns:", sorted(df.columns))

period              2017-2020  2021-2023
component variable                      
BIOPRO    LBXSCR         True       True
BMX       BMXBMI         True       True
          BMXWAIST       True       True
DEMO      RIAGENDR       True       True
          RIDAGEYR       True       True
DIQ       DIQ010         True       True
GHB       LBXGH          True       True
HDL       LBDHDD         True       True
MCQ       MCQ300A        True      False
          MCQ300C        True      False
PAQ       PAQ605         True      False
          PAQ620         True      False
          PAQ635         True      False
          PAQ650         True      False
          PAQ665         True      False
SMQ       SMQ020         True       True
          SMQ040         True       True
TCHOL     LBXTC          True       True

PAQ_L columns: ['PAD680', 'PAD790Q', 'PAD790U', 'PAD800', 'PAD810Q', 'PAD810U', 'PAD820', 'SEQN']

MCQ_L columns: ['AGQ030', 'MCQ010', 'MCQ035', 'MCQ040', 'MCQ050', 'MCQ053', 'MCQ

In [7]:
def find_processed_root():
    path = Path.cwd().resolve()
    for candidate in [path, *path.parents]:
        if (candidate / "data" / "processed").is_dir():
            return candidate
    raise FileNotFoundError("Could not find data/processed from " + str(path))

PROCESSED = find_processed_root() / "data" / "processed"
dev = pd.read_csv(PROCESSED / "cohort_2017-2020.csv")
val = pd.read_csv(PROCESSED / "cohort_2021-2023.csv")

PREDICTORS = ["age", "female", "bmi", "waist", "smoking", "ltpa_min_week",
              "sitting_min_day", "diabetes", "hba1c", "total_chol", "hdl", "creatinine"]

# 1. Missing values per predictor (%)
missing = pd.DataFrame({
    "2017-2020": dev[PREDICTORS].isna().mean() * 100,
    "2021-2023": val[PREDICTORS].isna().mean() * 100,
}).round(1)
print("Missing values (%)")
print(missing)

# 2. Numeric summary: look for impossible values in min / max
numeric = ["age", "bmi", "waist", "ltpa_min_week", "sitting_min_day",
           "hba1c", "total_chol", "hdl", "creatinine", "sbp", "dbp"]
print("\nNumeric summary, 2017-2020")
print(dev[numeric].describe(percentiles=[0.01, 0.5, 0.99]).T.round(1))
print("\nNumeric summary, 2021-2023")
print(val[numeric].describe(percentiles=[0.01, 0.5, 0.99]).T.round(1))

# 3. Categories
for col in ["female", "smoking", "diabetes"]:
    print(f"\n{col}")
    print(pd.DataFrame({
        "2017-2020": dev[col].value_counts(dropna=False),
        "2021-2023": val[col].value_counts(dropna=False),
    }))

Missing values (%)
                 2017-2020  2021-2023
age                    0.0        0.0
female                 0.0        0.0
bmi                    0.7        0.5
waist                  2.1        2.5
smoking                0.0        0.1
ltpa_min_week          0.2        0.9
sitting_min_day        0.6        0.6
diabetes               0.1        0.0
hba1c                  4.5        4.7
total_chol             6.2        8.4
hdl                    6.2        8.4
creatinine             6.6        9.4

Numeric summary, 2017-2020
                  count   mean    std   min     1%    50%     99%     max
age              4597.0   45.3   16.7  20.0   20.0   43.0    80.0    80.0
bmi              4565.0   28.9    7.1  14.6   17.7   27.7    52.2    78.6
waist            4499.0   97.5   16.6  57.9   67.8   95.9   144.6   178.0
ltpa_min_week    4590.0  256.5  467.5   0.0    0.0   50.0  1980.0  6860.0
sitting_min_day  4570.0  324.2  202.6   0.0   20.0  300.0   900.0  1320.0
hba1c          

In [9]:
for name, d in [("2017-2020", dev), ("2021-2023", val)]:
    x = d["ltpa_min_week"].dropna()
    print(f"{name}:  any activity = {(x > 0).mean():.1%}   "
          f">=150 min/week = {(x >= 150).mean():.1%}")

2017-2020:  any activity = 52.9%   >=150 min/week = 39.2%
2021-2023:  any activity = 84.7%   >=150 min/week = 57.0%


In [10]:
checks = {
    "bmi < 15":               lambda d: d["bmi"] < 15,
    "bmi > 60":               lambda d: d["bmi"] > 60,
    "waist < 60":             lambda d: d["waist"] < 60,
    "sitting > 20 h/day":     lambda d: d["sitting_min_day"] > 1200,
    "sitting > 18 h/day":     lambda d: d["sitting_min_day"] > 1080,
    "hdl < 20":               lambda d: d["hdl"] < 20,
    "hba1c > 14":             lambda d: d["hba1c"] > 14,
    "creatinine > 3":         lambda d: d["creatinine"] > 3,
    "dbp < 40":               lambda d: d["dbp"] < 40,
}
rows = []
for label, rule in checks.items():
    rows.append({"rule": label,
                 "2017-2020": int(rule(dev).sum()),
                 "2021-2023": int(rule(val).sum())})
print(pd.DataFrame(rows).to_string(index=False))

              rule  2017-2020  2021-2023
          bmi < 15          3          3
          bmi > 60         12          4
        waist < 60          1          0
sitting > 20 h/day          1          1
sitting > 18 h/day          4          6
          hdl < 20          2          0
        hba1c > 14          6          3
    creatinine > 3          3          0
          dbp < 40          0          3
